In [0]:
# Nomes do catalog e das camadas
catalog = "cinedata"

bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"

landing_path = f"/Volumes/{catalog}/bronze/landing"

In [0]:
# Caminhos de cada tabela
path_info = f"{landing_path}/movies_info_TMDB_IMDB.csv"
path_financials = f"{landing_path}/movies_financials_IMDB_TMDB.csv"
path_metrics = f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
path_credits = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"
path_reviews = f"{landing_path}/movies_reviews.csv"

# Leitura dos arquivos
df_info = spark.read.csv(path_info, header=True, inferSchema=True)
df_financials = spark.read.csv(path_financials, header=True, inferSchema=True)
df_metrics = spark.read.csv(path_metrics, header=True, inferSchema=True)
df_credits = spark.read.csv(path_credits, header=True, inferSchema=True)
df_reviews = spark.read.csv(path_reviews, header=True, inferSchema=True)

In [0]:
from pyspark.sql import functions as F

# Escrita das tabelas com timestamp
df_info\
    .withColumn("ingestion_datetime", F.current_timestamp())\
    .write.format("delta").mode("append")\
    .saveAsTable(f"{bronze_schema}.tb_movies_info")

df_financials\
    .withColumn("ingestion_datetime", F.current_timestamp())\
    .write.format("delta").mode("append")\
    .saveAsTable(f"{bronze_schema}.tb_movies_financials")

df_metrics\
    .withColumn("ingestion_datetime", F.current_timestamp())\
    .write.format("delta").mode("append")\
    .saveAsTable(f"{bronze_schema}.tb_movies_metrics")

df_credits\
    .withColumn("ingestion_datetime", F.current_timestamp())\
    .write.format("delta").mode("append")\
    .saveAsTable(f"{bronze_schema}.tb_credits_and_tags")

df_reviews\
    .withColumn("ingestion_datetime", F.current_timestamp())\
    .write.format("delta").mode("append")\
    .saveAsTable(f"{bronze_schema}.tb_movies_reviews")

In [0]:
from datetime import date, timedelta

# Formatação da data para extrair a cotação do dólar
dbutils.widgets.text("data_inicio", (date.today() - timedelta(days=7)).strftime("%m-%d-%Y"), "Data início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", date.today().strftime("%m-%d-%Y"), "Data fim (MM-DD-YYYY)")

data_inicio_formatada = dbutils.widgets.get("data_inicio")
data_fim_formatada = dbutils.widgets.get("data_fim")

In [0]:
# Extrai a cotação da API
import requests, json

url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'&"
    "$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

# Faz o request e checa erro
res = requests.get(url, timeout=30)
res.raise_for_status()

registros = res.json()["value"]

if not registros:
    raise ValueError("API sem cotações no período informado.")

print(registros)

[{'cotacaoCompra': 5.1803, 'dataHoraCotacao': '2026-09-30 13:11:44.783262'}, {'cotacaoCompra': 5.2073, 'dataHoraCotacao': '2026-10-01 13:10:35.4469'}, {'cotacaoCompra': 5.2232, 'dataHoraCotacao': '2026-10-02 13:03:16.256632'}, {'cotacaoCompra': 4.9853, 'dataHoraCotacao': '2026-10-05 13:07:36.558602'}, {'cotacaoCompra': 4.9692, 'dataHoraCotacao': '2026-10-06 13:03:21.267287'}, {'cotacaoCompra': 4.9929, 'dataHoraCotacao': '2026-10-07 13:05:23.249063'}]


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
# Criação da tabela de cotação

# schema da tabela
schema = StructType([
    StructField("dataHoraCotacao", StringType()),
    StructField("cotacaoCompra", DoubleType())
])

# Informações do json
linhas = [(r["dataHoraCotacao"], r["cotacaoCompra"]) for r in registros]

df_cotacao = spark.createDataFrame(linhas, schema)

# Escreve a tabela
df_cotacao\
    .withColumn("ingestion_datetime", F.current_timestamp())\
    .write.format("delta").mode("append")\
    .saveAsTable(f"{bronze_schema}.tb_cotacao_dolar")